# 02 The floor

The floor is the planner with no model in it. It reads the question with rules, finds the measure,
writes the SQL from catalog/measures.yaml and drafts a sentence from the rows. It exists because a
free tier can disappear in a week, and a product that only works with a key is a demo.

Two things here: how the floor does on the golden set, read from the committed results, and the
retriever it uses, with the dead end that lost to it.

In [1]:
from pathlib import Path
import os, sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ.setdefault("READBACK_ROOT", str(ROOT))
sys.path.insert(0, str(ROOT))

In [2]:
import json
card = json.loads((ROOT / "results" / "evals" / "scorecard.json").read_text())
floor = next(r for r in card["test"]["rows"] if r["provider"] == "floor")
{"passed": floor["passed"], "graded": floor["graded"], "readback_rate": floor["readback_rate"]}

{'passed': 98, 'graded': 100, 'readback_rate': 1.0}

In [3]:
{f: (v["passed"], v["n"]) for f, v in floor["families"].items()}

{'comparison': (17, 17),
 'definition': (14, 16),
 'lookup': (15, 15),
 'multi': (10, 10),
 'refusal': (10, 10),
 'top_k': (17, 17),
 'trend': (15, 15)}

The two misses are both definitions: the floor quotes the definition the lexical index ranks first,
and for two questions the build's own document words the term differently from the question. The
retrieval depth test in the scorecard moved the depth from three to eight and did not change either
miss, so it is wording, not depth.

## The retriever

Every non definition question in the golden set names its answer column in its SQL. The test is
recall at five: is that column among the first five column documents the retriever returns for the
question text.

In [4]:
import re, yaml
from readback_catalog.index import Index, Synonyms
from readback_catalog.tokens import tokens

index = Index.read(ROOT / "catalog" / "index.parquet")
synonyms = Synonyms(yaml.safe_load((ROOT / "catalog" / "synonyms.yaml").read_text())["synonyms"])
pattern = re.compile(r"\((?:\w+\()?(\w+)\)?\)\s+as\s+value\s+from\s+(\w+)\.(\w+)", re.I)

cases = []
for path in sorted((ROOT / "evals" / "golden").glob("*.yaml")):
    for q in yaml.safe_load(path.read_text())["questions"]:
        if q["family"] in {"definition", "refusal", "multi"} or not q.get("claims"):
            continue
        m = pattern.search(q["claims"][0].get("sql") or "")
        if m:
            column, wh, table = m.group(1), m.group(2), m.group(3)
            cases.append((q["question"], wh, f"{wh}.{table.removeprefix('mart_')}.{column}", f"{wh}.{table}.{column}"))
len(cases)

96

In [5]:
def hit(found, case):
    return any(h in (case[2], case[3]) or h.endswith("." + case[2].split(".")[-1]) for h in found)

def lexical(case, expand):
    q = synonyms.expand(case[0]) if expand else tokens(case[0])
    return [h.doc.id for h in index.search(q, 5, warehouse=case[1], kinds=["column"])]

plain = sum(hit(lexical(c, False), c) for c in cases) / len(cases)
with_synonyms = sum(hit(lexical(c, True), c) for c in cases) / len(cases)
round(plain, 3), round(with_synonyms, 3)

(0.594, 0.615)

## The dead end: an embedding retriever

The plan was an embedding retriever. It needed a key: every free embedding endpoint wants one, and
the floor must run with none, so it could not be the floor's retriever whatever it scored. It was
measured anyway with the keyless stand in below, a character trigram vector with cosine similarity,
the closest thing to a dense retriever that runs offline. It lost.

In [6]:
from collections import Counter
import math

columns = [d for d in index.docs if d.kind == "column"]

def grams(text):
    t = " " + " ".join(text.lower().split()) + " "
    return Counter(t[i:i + 3] for i in range(len(t) - 2))

df = Counter(g for d in columns for g in set(grams(d.text)))
N = len(columns)

def vector(text):
    g = grams(text)
    return {k: v * math.log(N / (1 + df.get(k, 0))) for k, v in g.items()}

def norm(v):
    return math.sqrt(sum(x * x for x in v.values())) or 1.0

vectors = [(d, vector(d.text)) for d in columns]
norms = {d.id: norm(v) for d, v in vectors}

def dense(case):
    q = vector(case[0]); nq = norm(q)
    scored = [(sum(q.get(k, 0) * w for k, w in v.items()) / (nq * norms[d.id]), d.id) for d, v in vectors if d.warehouse == case[1]]
    return [i for _, i in sorted(scored, key=lambda s: (-s[0], s[1]))[:5]]

trigram = sum(hit(dense(c), c) for c in cases) / len(cases)
{"lexical": round(plain, 3), "lexical with synonyms": round(with_synonyms, 3), "trigram cosine": round(trigram, 3)}

{'lexical': 0.594, 'lexical with synonyms': 0.615, 'trigram cosine': 0.5}

Recall at five is modest for all three, because many questions name a measure in words no column
uses; that is what describe and the synonym table are for, and the floor itself does not depend on
it, since it reads its measures from catalog/measures.yaml. As a ranking, though, the order is
clear, and the lexical index with the synonym table is the retriever. It needs no key, it is deterministic
under a fixed tie break, and when it misses, the miss is a word that can be added to
catalog/synonyms.yaml and reviewed in a diff. A dense retriever's miss is a number in a vector.